# Customer Churn Prediction

This notebook performs **Cross-Validation and Hyperparameter Tuning** using classic ML models:
- Logistic Regression
- Random Forest
- Support Vector Machine (SVM)

Dataset: Telco Customer Churn (Kaggle)

## 1. Import Libraries

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import classification_report, roc_auc_score


## 2. Load Dataset

In [2]:

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Fix TotalCharges data type issue
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Handle missing values created due to coercion
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


/tmp/ipython-input-523219390.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


errors="coerce" means:

Any value that cannot be converted to a number will be replaced with NaN (missing value).


| Option              | Behavior                            |
| ------------------- | ----------------------------------- |
| `"raise"` (default) | Throws an error if conversion fails |
| `"coerce"`          | Invalid values → `NaN`              |
| `"ignore"`          | Leaves data unchanged               |


## 3. Preprocessing

In [5]:

df.drop("customerID", axis=1, inplace=True)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop("Churn", axis=1)
y = df["Churn"]

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols)
])


## 4. Cross-Validation Strategy

In [6]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 5. Logistic Regression – Hyperparameter Tuning

Parameters to Tune

C → Regularization strength

penalty → Type of regularization

In [7]:

log_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

param_log = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"]
}

grid_log = GridSearchCV(log_model, param_log, cv=cv, scoring="f1", n_jobs=-1)
grid_log.fit(X, y)

grid_log.best_params_, grid_log.best_score_


({'model__C': 10, 'model__penalty': 'l2'}, np.float64(0.6007585540324282))

log_model → The base machine learning model to be tuned

param_log → Dictionary of hyperparameters to try

cv=cv → Number of cross-validation folds

scoring="f1" → Metric used to select the best model

n_jobs=-1 → Uses all CPU cores for faster computation

## 6. Random Forest – Hyperparameter Tuning

Parameters to Tune

n_estimators → Number of trees

max_depth → Tree depth

min_samples_split → Overfitting control

If cv = 5, GridSearchCV computes:

Fold 1 F1 = 0.82

Fold 2 F1 = 0.85

Fold 3 F1 = 0.84

Fold 4 F1 = 0.83

Fold 5 F1 = 0.87

best_score_
=
0.82
+
0.85
+
0.84
+
0.83
+
0.87
5
=

best_score_=
5
0.82+0.85+0.84+0.83+0.87
	​

=0.842

Higher value = better model

In [8]:

rf_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

param_rf = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(rf_model, param_rf, cv=cv, scoring="f1", n_jobs=-1)
grid_rf.fit(X, y)

grid_rf.best_params_, grid_rf.best_score_


({'model__max_depth': 10,
  'model__min_samples_split': 2,
  'model__n_estimators': 200},
 np.float64(0.5743397878172051))

## 7. SVM – Hyperparameter Tuning

Parameters to Tune

C → Margin softness

kernel → Linear vs non-linear

gamma → Influence of points (RBF kernel)

In [9]:

svm_model = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(probability=True))
])

param_svm = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV(svm_model, param_svm, cv=cv, scoring="f1", n_jobs=-1)
grid_svm.fit(X, y)

grid_svm.best_params_, grid_svm.best_score_


({'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'linear'},
 np.float64(0.5856982591006066))

## 8. Final Model Evaluation (Best Model: Random Forest)

In [10]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_model = grid_rf.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.53      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409

ROC-AUC: 0.8432832157896099


| ROC-AUC Score | What it tells you          |
| ------------- | -------------------------- |
| **1.0**       | Perfect class separation   |
| **0.9+**      | Excellent model            |
| **0.8 – 0.9** | Good model                 |
| **0.7 – 0.8** | Acceptable                 |
| **0.5**       | No discrimination (random) |
| **< 0.5**     | Model is misleading        |


In [11]:
best_log_model = grid_log.best_estimator_
best_log_model.fit(X_train, y_train)

y_pred_log = best_log_model.predict(X_test)
y_prob_log = best_log_model.predict_proba(X_test)[:,1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_log))
print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, y_prob_log))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409

Logistic Regression ROC-AUC: 0.8411661370740655


## Evaluate SVM Model



In [12]:
best_svm_model = grid_svm.best_estimator_
best_svm_model.fit(X_train, y_train)

y_pred_svm = best_svm_model.predict(X_test)
y_prob_svm = best_svm_model.predict_proba(X_test)[:,1]

print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print("SVM ROC-AUC:", roc_auc_score(y_test, y_prob_svm))

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.62      0.53      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

SVM ROC-AUC: 0.8251026892970627


## 9. Conclusion
Random Forest achieved the best performance after hyperparameter tuning and cross-validation.